# Banking Transactions Data Pipeline

This notebook ingests a raw banking transaction CSV, profiles it, cleans it, validates it, transforms it, aggregates it and saves curated outputs.

Every code line has a comment explaining what it does. See `README.md` for the list of problems and solutions.

In [ ]:
import pandas as pd  # load pandas, the main library for tabular data
import numpy as np  # load numpy for vectorised numeric operations such as np.where
import os  # load the os module to work with folders

In [ ]:
INPUT_FILE = 'bank_transactions.csv'  # path to the raw banking CSV file
OUTPUT_DIR = 'output'  # folder where all result files will be saved
os.makedirs(OUTPUT_DIR, exist_ok=True)  # create the output folder if it does not already exist

## 1. Ingest raw data
Every column is read as text so that pandas does not silently change values such as dates or amounts.

In [ ]:
raw = pd.read_csv(INPUT_FILE, dtype=str)  # read the CSV with every column as text
print('Rows, columns:', raw.shape)  # print the number of rows and columns
raw.head()  # display the first five rows to check the load

## 2. Profile the data
Count missing values, duplicates and the distinct values of key columns to see what needs fixing.

In [ ]:
print(raw.isna().sum())  # count missing values in each column
print('Duplicate transaction_id rows:', raw.duplicated(subset='transaction_id').sum())  # count rows whose ID already appeared earlier
print('Duplicate content rows (all fields except transaction_id):', raw.drop(columns='transaction_id').duplicated().sum())  # count rows that copy another row's data even when the ID differs
print(raw['transaction_type'].value_counts(dropna=False))  # show each transaction_type value, including dirty ones such as 'debit '
print(raw['currency'].value_counts(dropna=False))  # show each currency value, including dirty ones such as 'mur'
print(raw['status'].value_counts(dropna=False))  # show how many SUCCESS and FAILED rows exist

## 3. Clean the data
Fix text formatting, convert types, handle missing categories and remove duplicates.

In [ ]:
df = raw.copy()  # make a copy so the raw data stays unchanged
df['transaction_type'] = df['transaction_type'].str.strip().str.upper()  # remove spaces and convert 'debit ' to 'DEBIT'
df['currency'] = df['currency'].str.strip().str.upper()  # convert 'mur' to 'MUR' so codes match
df['merchant_category'] = df['merchant_category'].fillna('UNKNOWN').str.strip()  # replace an empty category with 'UNKNOWN'
df['status'] = df['status'].str.strip().str.upper()  # standardise status text to upper case
df['account_id'] = df['account_id'].str.strip()  # remove stray spaces from account IDs (missing values stay missing)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')  # convert amount to a number; empty or invalid text becomes NaN
df['balance_after'] = pd.to_numeric(df['balance_after'], errors='coerce')  # convert balance to a number the same way
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='%Y-%m-%d', errors='coerce')  # convert to a date; impossible dates such as 2026-02-30 become NaT
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'transaction_id'], keep='first')  # remove rows that repeat every field except transaction_id, keeping the first copy

## 4. Validate with business rules
Each rule creates a flag column. Rows with any flag go to quarantine for review, and the rest are clean.

In [ ]:
df['issue_missing_account'] = df['account_id'].isna()  # flag rows that have no account_id
df['issue_missing_amount'] = df['amount'].isna()  # flag rows that have no amount
df['issue_bad_date'] = df['transaction_date'].isna()  # flag rows whose date is missing or invalid
df['issue_negative_amount'] = df['amount'] < 0  # flag amounts below zero, because a clean feed uses positive amounts
df['issue_failed_status'] = df['status'] != 'SUCCESS'  # flag transactions that did not succeed
issue_cols = [c for c in df.columns if c.startswith('issue_')]  # collect the names of all flag columns
df['has_issue'] = df[issue_cols].any(axis=1)  # True when at least one rule failed for that row
quarantine = df[df['has_issue']].copy()  # rows that need human review
clean = df[~df['has_issue']].copy()  # rows that passed every rule
print(f'Clean rows: {len(clean)}  |  Quarantined rows: {len(quarantine)}')  # print how many rows went to each group

In [ ]:
quarantine[['transaction_id'] + issue_cols].head(10)  # show which rule each quarantined row failed

## 5. Transform
Add derived columns that analytics needs: a signed amount, a month label, an hour and a large-transaction flag.

In [ ]:
clean['signed_amount'] = np.where(clean['transaction_type'] == 'CREDIT', clean['amount'], -clean['amount'])  # positive for money in, negative for money out
clean['txn_month'] = clean['transaction_date'].dt.to_period('M').astype(str)  # year-month label such as 2026-09
clean['txn_hour'] = pd.to_datetime(clean['transaction_time'], format='%H:%M:%S').dt.hour  # extract the hour of the day from the time
clean['is_large_txn'] = clean['amount'] >= 10000  # flag transactions of 10,000 or more for AML review
clean.head()  # show the transformed rows

## 6. Data quality checks
Assertions stop the notebook if the clean data breaks a key rule.

In [ ]:
assert clean['transaction_id'].is_unique, 'Duplicate transaction IDs remain in clean data'  # fail loudly if any duplicate ID is still present
assert clean['amount'].gt(0).all(), 'Clean data contains zero or negative amounts'  # fail if any amount is zero or below zero
assert clean['account_id'].notna().all(), 'Clean data contains missing account IDs'  # fail if any account ID is missing
print('All data quality checks passed')  # confirm that every check succeeded

## 7. Aggregate
Build summaries for reporting: monthly net flow per account, channel volumes, and the list of large transactions.

In [ ]:
monthly_account = clean.groupby(['account_id', 'txn_month'], as_index=False).agg(net_flow=('signed_amount', 'sum'), txn_count=('transaction_id', 'count'))  # total net flow and number of transactions per account per month
channel_summary = clean.groupby('channel', as_index=False).agg(txn_count=('transaction_id', 'count'), total_value=('amount', 'sum'))  # number and total value of transactions per channel
large_txns = clean[clean['is_large_txn']][['transaction_id', 'account_id', 'transaction_date', 'amount', 'channel']]  # list of large transactions for compliance review
monthly_account  # display the monthly account summary

## 8. Save outputs
Write curated data, quarantine data and summaries to the output folder as CSV files.

In [ ]:
clean.to_csv(os.path.join(OUTPUT_DIR, 'curated_transactions.csv'), index=False)  # save clean, curated transactions without the row index
quarantine.to_csv(os.path.join(OUTPUT_DIR, 'quarantine_transactions.csv'), index=False)  # save rows that need review, with their flags
monthly_account.to_csv(os.path.join(OUTPUT_DIR, 'monthly_account_summary.csv'), index=False)  # save the monthly account summary
channel_summary.to_csv(os.path.join(OUTPUT_DIR, 'channel_summary.csv'), index=False)  # save the channel summary
large_txns.to_csv(os.path.join(OUTPUT_DIR, 'large_transactions.csv'), index=False)  # save the large transactions list
print('Files saved to folder:', OUTPUT_DIR)  # confirm where the files were written